Thie notebook creates simulations to contextualize the FN and FP rate for the hypothesis "cre of interest is significantly different from minP".
- negative simulations (cre_oi=minP) for quantifying the FP rate.
- positive simulations (cre_oi!=minP) for quantifying FN rate.

# Setup

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

%load_ext autoreload
%autoreload 2

2026-01-14 12:25:43.258032: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-14 12:25:43.312822: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [3]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=4:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=4)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

In [4]:
data_root="/nfs/roberts/project/pi_skr2/shared/tabula_data"

# Creating artificial libraries

In [ ]:
#making up the CREs
spread_gt,spread_hypothesis=scm.activity_spread(
    cell_types=list(scm.SHENDURE_BOUNDS.cells_per_cell_type.keys()),
    minimum=scm.SHENDURE_BOUNDS.min_mpra_umi,
    maximum=scm.SHENDURE_BOUNDS.max_mpra_umi,
    minp_value=scm.SHENDURE_BOUNDS.reference_activity,
    total=100,
    frac_active=0.5,
    ct_specificity=.2)

libraries=[scm.simulate_library(CREs=spread_gt["cre_id"],
                 library_model=scm.SHENDURE_BOUNDS.library_model)
                 for i in range(5)]

In [ ]:
spread_hypothesis.to_tsv(f"{data_root}/pow_sim_2026-01-03_hypo.tsv")

# Creating sim

In [ ]:
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client,
                            libraries=libraries,
                            library_mapping="corresponding",
                            n_sims=5,
                            experiment_bounds=scm.SHENDURE_BOUNDS,
                            ground_truth=spread_gt)

In [ ]:
sim.gamut()

In [ ]:
sim.save()

# Fit orthos

In [ ]:
#loading back into mem, assuming notebook shut down after last task
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client)

In [ ]:
sim.fit_orthos()

In [ ]:
sim.save()

# Wald precompute: sandwich

In [ ]:
#loading back into mem, assuming notebook shut down after last task
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client)

In [ ]:
sim.precompute_wald(cov_method="sandwich")

In [ ]:
sim.save()

# Wald precompute: opg

In [ ]:
#loading back into mem, assuming notebook shut down after last task
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client)

In [ ]:
sim.precompute_wald(cov_method="opg")

In [ ]:
sim.save()

# Hypothesis testing

In [ ]:
#loading back into mem, assuming notebook shut down after last task
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client)

Add a basic hypothesis set

In [ ]:
spread_hypothesis=scm.HypothesisSet.from_tsv(f"{data_root}/pow_sim_2026-01-03_hypo.tsv")

In [ ]:
sim.add_hypothesis_set(name="spread",hypotheses=spread_hypothesis)

Run mwu

In [ ]:
sim.mwu("spread")

In [ ]:
sim.save()

In [ ]:
sim.wald("spread")

In [ ]:
sim.wald("spread",cov_method="opg")

In [ ]:
sim.save()

# Summary

In [5]:
#loading back into mem, assuming notebook shut down after last task
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2026-01-03",
                            client=client)

scMPRAforge: INFO: 'state.parquet' found for 'pow_sim_2026-01-03', loading.


In [6]:
sim.list_tests()

{'spread': ['mwu', 'wald_opg', 'wald_sandwich']}

In [10]:
sim._merge_in_ground_truth(hypothesis_set_name="spread",test_type="mwu",index=0)

,comparison_CRE,reference_CRE,comparison_cell_type,reference_cell_type,meta,test_statistic,p_value,fold_change,flattened,ref_mean,comp_mean,test_type,bh_p,comparison_truth,reference_truth,gt_effect_size,gt_null,reject_null
0,inactive_0,reference,Cardiomyocytes,Cardiomyocytes,<NA>,143989.0,6.477094e-01,1.122470,True,0.012772305202551903,0.015561231975419032,mwu,9.504807e-01,0.019311,0.019311,1.000000,True,False
1,inactive_0,reference,EpiblastPrimitiveStreak,EpiblastPrimitiveStreak,<NA>,3068977.5,6.391818e-01,0.949530,True,0.015259316922416666,0.013984484013818133,mwu,9.466363e-01,0.019311,0.019311,1.000000,True,False
2,inactive_0,reference,ExEndodermParietal,ExEndodermParietal,<NA>,6038647.0,1.661610e-02,1.228339,True,0.010886331819568218,0.015655493427906952,mwu,1.281747e-01,0.019311,0.019311,1.000000,True,False
3,inactive_0,reference,ExEndodermVisceral,ExEndodermVisceral,<NA>,2902892.5,3.929909e-01,1.118367,True,0.01155729843120934,0.014108971780954382,mwu,8.563590e-01,0.019311,0.019311,1.000000,True,False
4,inactive_0,reference,Haematoendothelial,Haematoendothelial,<NA>,322462.5,8.245754e-01,1.028264,True,0.011435634447625631,0.012041496576213007,mwu,9.791021e-01,0.019311,0.019311,1.000000,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5495,active_45,active_45,reference,SurfaceEctoderm,<NA>,38833808.5,6.749567e-01,1.000359,True,65.30445780621022,65.32790420122166,mwu,9.543213e-01,346.477666,346.477666,1.000000,True,False
5496,active_46,active_46,reference,SurfaceEctoderm,<NA>,30701340.5,2.723733e-01,1.029332,True,6.402449682345704,6.590537346732208,mwu,7.810531e-01,22.699749,22.699749,1.000000,True,False
5497,active_47,active_47,reference,SurfaceEctoderm,<NA>,21858158.0,7.250828e-44,0.601216,True,54.39607939584091,32.69979082869138,mwu,7.290595e-43,161.621961,270.766683,0.596905,False,True
5498,active_48,active_48,reference,SurfaceEctoderm,<NA>,12639686.0,1.576764e-01,0.936356,True,60.19731713091773,56.36546564712449,mwu,6.436762e-01,321.033991,321.033991,1.000000,True,False


# Shutdown

In [ ]:
client.close()
cluster.close()